In [1]:
# ============================================================
# 🎯 AI Interview Preparation Bot — Google Colab
# ============================================================
# Install dependency
!pip install anthropic -q

import anthropic
import textwrap
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

# ── Config ──────────────────────────────────────────────────
API_KEY = "YOUR_API_KEY_HERE"   # ← paste your Anthropic key

ROLES = [
    "Software Engineer",
    "Data Scientist",
    "Product Manager",
    "Frontend Engineer",
    "ML Engineer",
    "DevOps / SRE",
]

SYSTEM_PROMPT = """You are an expert technical interviewer with 15+ years of experience hiring at top tech companies.

Your job is to conduct a realistic mock interview based on the candidate's chosen role.

Guidelines:
- Ask ONE question at a time (behavioural, technical, or system design — vary them naturally).
- After the candidate answers, give brief, constructive feedback (2-3 sentences max): what was strong, what could improve.
- Then ask the NEXT question.
- Keep a natural conversational tone — encouraging but honest.
- After 5 questions, give a short overall summary with a score out of 10 and top 3 improvement areas.

Start by warmly greeting the candidate and asking your first question immediately."""

# ── State ────────────────────────────────────────────────────
client = anthropic.Anthropic(api_key=API_KEY)
conversation_history = []
question_count = 0
selected_role = ROLES[0]

# ── Helpers ──────────────────────────────────────────────────
def chat(user_message: str) -> str:
    global question_count
    conversation_history.append({"role": "user", "content": user_message})

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        system=SYSTEM_PROMPT + f"\n\nThe candidate is interviewing for: {selected_role}",
        messages=conversation_history,
    )

    assistant_message = response.content[0].text
    conversation_history.append({"role": "assistant", "content": assistant_message})
    question_count += 1
    return assistant_message

def wrap(text: str, width: int = 90) -> str:
    return "\n".join(textwrap.fill(line, width) for line in text.splitlines())

def display_bubble(role: str, text: str):
    color  = "#1e3a5f" if role == "Interviewer" else "#2d5a27"
    label  = "🤖 Interviewer" if role == "Interviewer" else "🙋 You"
    html = f"""
    <div style="margin:8px 0; font-family:monospace;">
      <span style="font-size:11px; color:{color}; font-weight:bold;">{label}</span>
      <div style="background:{'#e8f0fe' if role=='Interviewer' else '#e8f5e9'};
                  border-left:4px solid {color};
                  padding:10px 14px; margin-top:4px;
                  border-radius:0 8px 8px 0;
                  white-space:pre-wrap; font-size:14px; color:#1a1a1a;">
{text}
      </div>
    </div>"""
    display(HTML(html))

# ── Widget UI ─────────────────────────────────────────────────
def build_ui():
    global selected_role, conversation_history, question_count

    # Role selector
    role_dd = widgets.Dropdown(
        options=ROLES, value=ROLES[0],
        description="Role:",
        style={"description_width": "initial"},
        layout=widgets.Layout(width="320px"),
    )

    start_btn = widgets.Button(
        description="Start Interview",
        button_style="success",
        icon="play",
        layout=widgets.Layout(width="160px"),
    )

    reset_btn = widgets.Button(
        description="Reset",
        button_style="warning",
        icon="refresh",
        layout=widgets.Layout(width="100px"),
    )

    answer_box = widgets.Textarea(
        placeholder="Type your answer here…",
        layout=widgets.Layout(width="100%", height="100px"),
        disabled=True,
    )

    send_btn = widgets.Button(
        description="Send Answer",
        button_style="primary",
        icon="send",
        disabled=True,
        layout=widgets.Layout(width="140px"),
    )

    chat_output = widgets.Output()
    status      = widgets.Label(value="Choose a role and click Start Interview.")

    # ── Callbacks ────────────────────────────────────────────
    def on_start(_):
        global selected_role, conversation_history, question_count
        selected_role      = role_dd.value
        conversation_history = []
        question_count     = 0
        answer_box.disabled  = False
        send_btn.disabled    = False
        start_btn.disabled   = True

        with chat_output:
            clear_output()
            display(HTML(f"<h3 style='font-family:monospace;color:#1e3a5f'>📋 Mock Interview — {selected_role}</h3>"))

        status.value = "Interview started! Answer the question below."
        reply = chat(f"Start the interview for a {selected_role} position.")
        with chat_output:
            display_bubble("Interviewer", reply)

    def on_send(_):
        user_text = answer_box.value.strip()
        if not user_text:
            return
        answer_box.value   = ""
        send_btn.disabled  = True

        with chat_output:
            display_bubble("You", user_text)

        status.value = "Interviewer is thinking…"
        reply = chat(user_text)

        with chat_output:
            display_bubble("Interviewer", reply)

        # Disable after summary (≥5 exchanges)
        if question_count >= 5:
            answer_box.disabled = True
            status.value = "Interview complete! Click Reset to start again."
        else:
            send_btn.disabled = False
            status.value = f"Question {question_count}/5 — type your answer and click Send."

    def on_reset(_):
        global conversation_history, question_count
        conversation_history = []
        question_count       = 0
        answer_box.disabled  = True
        send_btn.disabled    = True
        start_btn.disabled   = False
        answer_box.value     = ""
        status.value         = "Choose a role and click Start Interview."
        with chat_output:
            clear_output()

    start_btn.on_click(on_start)
    send_btn.on_click(on_send)
    reset_btn.on_click(on_reset)

    # ── Layout ───────────────────────────────────────────────
    top_bar = widgets.HBox([role_dd, start_btn, reset_btn],
                           layout=widgets.Layout(gap="12px", align_items="center"))
    input_row = widgets.HBox([answer_box, send_btn],
                             layout=widgets.Layout(gap="8px", align_items="flex-end"))

    display(HTML("<h2 style='font-family:monospace'>🎯 AI Interview Prep Bot</h2>"))
    display(top_bar)
    display(widgets.HTML("<hr style='margin:8px 0'>"))
    display(chat_output)
    display(widgets.HTML("<hr style='margin:8px 0'>"))
    display(input_row)
    display(status)

build_ui()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 929.8/929.8 kB 13.8 MB/s eta 0:00:00


HTML(value="<hr style='margin:8px 0'>")

Output()

HTML(value="<hr style='margin:8px 0'>")

Label(value='Choose a role and click Start Interview.')